<a href="https://colab.research.google.com/github/goksungurel/CE_477/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

URL_2024 = "https://acikveri.bizizmir.com/dataset/95bec47b-a29c-4ab0-a070-434334588da8/resource/23bfeebe-1370-49da-aed0-5573cf3dd612/download/2024-yili-yangin-mudahale-cizelgesi.xlsx"
URL_2025 = "https://acikveri.bizizmir.com/dataset/95bec47b-a29c-4ab0-a070-434334588da8/resource/d9bfb046-f67b-4fb3-bc62-43068895c19f/download/2025-yili-yangin-mudahale-cizelgesi.xlsx"
d24 = pd.read_excel(URL_2024); d24["YIL"] = 2024
d25 = pd.read_excel(URL_2025); d25["YIL"] = 2025
df = pd.concat([d24, d25], ignore_index=True)
print(df.shape)
df.head()

Setup (rename + type fixes)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, os
os.makedirs("figures", exist_ok=True)

# English column names
df = df.rename(columns={
    "TARIH": "date", "YANGIN_TURU": "fire_type", "YANGIN_SEBEBİ": "fire_cause",
    "YANGIN_SONUCU": "fire_result", "YAPI_SEKLI": "building_type",
    "ERKEK_VEFAT": "male_deaths", "KADIN_VEFAT": "female_deaths", "ITFAIYECI_VEFAT": "firefighter_deaths",
    "ERKEK_YARALI": "male_injured", "KADIN_YARALI": "female_injured", "ITFAIYECI_YARALI": "firefighter_injured",
    "BUYUK_BAS_VEFAT": "cattle_deaths", "KUCUK_BAS_VEFAT": "sheep_goat_deaths",
    "KUMES_HAYVANI_VEFAT": "poultry_deaths", "EVCIL_HAYVAN_VEFAT": "pet_deaths",
    "DIGER HAYVAN": "other_animal_deaths", "VARIS_SURESI (DAK.)": "arrival_time",
    "KULLANILAN_KOPUK_MIKTARI (KG)": "foam_kg", "KULLANILAN_SU_MIKTARI (m³)": "water_m3",
    "KULLANILAN_KURU_KIMYEVİ_TOZ_MIKTARI (KG)": "dry_powder_kg",
    "EKIPLERİN_CIKIS_SAATI": "dispatch_time", "ILCE": "district",
    "ADRES_BOLGESI": "region", "YIL": "year"})

# type fixes + derived attributes
df["arrival_min"] = pd.to_timedelta(df["arrival_time"] + ":00").dt.total_seconds() / 60
df["dispatch_hour"] = df["dispatch_time"].str[:2].astype(int)
df["other_animal_deaths"] = pd.to_numeric(df["other_animal_deaths"], errors="coerce")
df["month"] = df["date"].dt.month

# English category labels
result_map = {
    "BAŞLANGIÇTA SÖNDÜRÜLEN": "Extinguished early",
    "KISMEN YANARAK KURTARILAN": "Partially burned",
    "TAMAMEN YANAN": "Completely burned"}
region_map = {"KENT MERKEZİ": "Urban", "KIRSAL BÖLGE": "Rural", "İL DIŞI": "Outside province"}

for col, m in [("fire_result", result_map), ("region", region_map)]:
    df[col] = df[col].str.strip()
    unmapped = set(df[col].dropna().unique()) - set(m)
    if unmapped:
        print(f"{col} - sözlükte olmayan değerler:", unmapped)
    df[col] = df[col].map(m)

result_order = ["Extinguished early", "Partially burned", "Completely burned"]
df["fire_result"] = pd.Categorical(df["fire_result"], categories=result_order, ordered=True)
df["fire_type"] = df["fire_type"].replace({
    "OT": "Grass", "ÇÖP": "Garbage", "BİNA": "Building", "ARAÇ": "Vehicle", "İŞYERİ": "Workplace",
    "AĞAÇ": "Tree", "KARGILIK": "Reed bed", "ORMAN/FİDANLIK": "Forest/nursery",
    "BAĞ/BAHÇE": "Vineyard/garden", "TRAFO": "Transformer",
    "FABRİKA/ATÖLYE/İMALATHANE": "Factory/workshop", "KABLO/DİREK-ELEKT/TEL VB": "Cable/pole",
    "BARAKA/KULÜBE/YAŞAM KONTEYNERİ": "Shack/container", "ZEYTİNLİK": "Olive grove"})
df["fire_cause"] = df["fire_cause"].replace({
    "SİGARA/KİBRİT": "Cigarette/match", "AÇIK ATEŞ": "Open fire",
    "ELEKT. KISA DEV./ELEKT. KONTAĞI": "Electrical short circuit",
    "KIVILCIM(DİĞER)": "Spark (other)", "KIVILCIM(BACA)": "Spark (chimney)",
    "ORMAN İTF. TESPİT EDECEK": "To be determined by forestry", "ŞÜPHELİ": "Suspicious",
    "ISINMA/SÜRTÜNME": "Heating/friction", "OCAK ÜSTÜ YEMEK UNUTMA": "Unattended cooking",
    "SİRAYET": "Spread from nearby fire"})
df["building_type"] = df["building_type"].replace({
    "BETONARME": "Reinforced concrete", "DİĞER": "Other", "KAGİR": "Masonry", "ÇELİK": "Steel",
    "AHŞAP": "Timber", "PREFABRİK": "Prefabricated", "BAĞDADİ(TAŞ YAPI)": "Bagdadi (timber-frame)"})
print(df.shape)
df.dtypes

In [ ]:
# translate the remaining fire types and causes
df["fire_type"] = df["fire_type"].replace({
    "EKİN/SAMAN": "Crop/straw", "MAKİLİK/ÇALI": "Maquis/shrubland", "ANIZ": "Stubble",
    "DİĞER": "Other", "HURDA-AÇIK ALANDA": "Scrap (open area)", "LASTİK": "Tyre",
    "SİGORTA PANO-BİNADAN BAĞIMSIZ": "Electrical panel (detached)", "MOTOSİKLET": "Motorcycle",
    "DAM/AHIR/AĞIL/AMBAR": "Barn/stable/shed", "ODUNLUK/KÖMÜRLÜK": "Woodshed/coal store",
    "ODUN/KÖMÜR-AÇIK ALANDA": "Wood/coal (open area)", "AÇIK ALAN DEPO": "Open-air storage",
    "KERESTE/PALET VB": "Timber/pallets", "GÜBRE": "Manure/fertiliser", "GEMİ/TEKNE": "Ship/boat",
    "HURDALIK": "Scrapyard", "İNŞAAT ALANI": "Construction site", "İŞ MAKİNESİ": "Construction machinery",
    "PARK": "Park", "SERA": "Greenhouse", "KÜMES": "Poultry house",
    "REKLAM/İŞYERİ TABELASI/PANOSU VB": "Billboard/signboard", "ARAÇ (ELEKTRİKLİ)": "Electric vehicle",
    "ARAÇ-TEHLİKELİ MAD TAŞIYAN": "Hazmat vehicle", "MOTOSİKLET (ELEKTRİKLİ)": "Electric motorcycle",
    "PETROL İSTASYONU": "Petrol station", "TREN": "Train"})
df["fire_cause"] = df["fire_cause"].replace({
    "PARLAMA (YAĞ/KİMYASAL MAD.)": "Flash fire (oil/chemical)", "PARLAMA (AKARYAKIT MADDESİ)": "Flash fire (fuel)",
    "İHMAL/DİKKATSİZLİK": "Negligence", "DİĞER": "Other",
    "ÇOCUKLARIN ATEŞLE OYNAMASI": "Children playing with fire", "KASIT": "Intentional",
    "KIZIŞMA": "Spontaneous heating", "LPG/DOĞALGAZ": "LPG/natural gas",
    "KONDÜKSİYON (TEMAS İLE ISI TRANSFERİ)": "Conduction (contact heat transfer)",
    "DOĞAL OLAYLAR (YILDIRIM/GÜNEŞ VB)": "Natural causes (lightning/sun)",
    "KUNDAKLAMA/SABOTAJ": "Arson/sabotage", "KAZA SONUCU": "Accident", "HARARET": "Engine overheating",
    "KİMYASAL MADDE REAKSİYONU": "Chemical reaction",
    "RADYASYON (IŞINIM İLE ISI TRANSFERİ)": "Radiation (radiant heat transfer)",
    "KONVEKSİYON (HAVA İLE ISI TRANSFERİ)": "Convection (air heat transfer)",
    "OCAK/SOBA/KALORİFER KAZANI": "Stove/heater/boiler",
    "KONTROLLÜ ATEŞİN YANGINA DÖNÜŞMESİ": "Controlled burn got out of control",
    "ELEKT. KISA DEV. (TERMAL KAÇAK)": "Electrical (thermal runaway)", "TESPİT EDİLEMEDİ": "Undetermined",
    "İNFİLAK": "Explosion", "STATİK ELEKTRİK": "Static electricity", "PATLAYICI MADDE": "Explosive material"})
print(df["fire_type"].unique())

Missing values


In [ ]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_%"] = (missing["missing_count"] / len(df) * 100).round(2)
print(missing)
# building_type is only filled for structure fires:
print(df[df["building_type"].notna()]["fire_type"].value_counts())

Box plots


In [ ]:
numeric = ["arrival_min", "water_m3", "foam_kg", "dry_powder_kg"]
titles = ["Arrival time (min)", "Water used (m³)", "Foam used (kg)", "Dry chemical powder (kg)"]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col, t in zip(axes, numeric, titles):
    sns.boxplot(y=np.log10(1 + df[col].dropna()), ax=ax)
    ax.set_title(t); ax.set_ylabel("log10(1 + value)")
plt.tight_layout(); plt.savefig("figures/boxplot.png", dpi=300); plt.show()
print(df[numeric].describe().round(2))

Histograms (ordinal)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["fire_result"].value_counts(sort=False).plot(kind="bar", ax=axes[0], logy=True)
axes[0].set_title("Fire result (log scale)"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
axes[1].hist(df["dispatch_hour"], bins=24, edgecolor="white")
axes[1].set_title("Dispatch hour"); axes[1].set_xlabel("hour of day")
axes[2].hist(df["month"], bins=12, edgecolor="white")
axes[2].set_title("Month"); axes[2].set_xlabel("month")
for ax in axes: ax.set_ylabel("number of fires")
plt.tight_layout(); plt.savefig("figures/histogram.png", dpi=300); plt.show()

Mode/median

In [ ]:
categorical = ["fire_type", "fire_cause", "fire_result", "building_type", "district", "region"]
for c in categorical:
    print(f"{c}: mode = {df[c].mode()[0]}  ({df[c].value_counts().iloc[0]} records)")
codes = df["fire_result"].cat.codes
codes = codes[codes >= 0]  # Ignore NaN values (-1)
med = int(codes.quantile(0.5, interpolation="lower"))
print("fire_result median:", result_order[med])

Bar + stacked bar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["fire_type"].value_counts().head(10).plot(kind="barh", ax=axes[0]); axes[0].invert_yaxis()
axes[0].set_title("Top 10 fire types"); axes[0].set_xlabel("number of fires"); axes[0].set_ylabel("")
df["fire_cause"].value_counts().head(10).plot(kind="barh", ax=axes[1], color="tab:orange"); axes[1].invert_yaxis()
axes[1].set_title("Top 10 fire causes"); axes[1].set_xlabel("number of fires"); axes[1].set_ylabel("")
plt.tight_layout(); plt.savefig("figures/bar.png", dpi=300); plt.show()

top10 = df["fire_type"].value_counts().head(10).index
table = pd.crosstab(df["fire_type"], df["fire_result"], normalize="index").loc[top10] * 100
table.plot(kind="barh", stacked=True, figsize=(9, 5))
plt.xlabel("% of fires"); plt.ylabel(""); plt.title("Fire result by fire type")
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=3)
plt.tight_layout(); plt.savefig("figures/stacked_bar.png", dpi=300); plt.show()
print(table.round(1))

Scatter plots

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="arrival_min", y=np.log10(1 + df["water_m3"]),
                hue="fire_result", alpha=0.4, s=12)
plt.xlim(0, 60); plt.xlabel("Arrival time (min)"); plt.ylabel("log10(1 + water m³)")
plt.title("Arrival time vs water used")
plt.savefig("figures/scatter.png", dpi=300, bbox_inches="tight"); plt.show()

dist = df.groupby("district").agg(
    median_arrival=("arrival_min", "median"),
    damaged_pct=("fire_result", lambda x: (x != "Extinguished early").mean() * 100),
    n_fires=("date", "size"))
plt.figure(figsize=(8, 5))
plt.scatter(dist["median_arrival"], dist["damaged_pct"], s=dist["n_fires"] / 10, alpha=0.6)
for name, row in dist.iterrows():
    plt.annotate(name, (row["median_arrival"], row["damaged_pct"]), fontsize=7)
plt.xlabel("Median arrival time (min)"); plt.ylabel("% partially/completely burned")
plt.title("Districts: arrival time vs damage rate (size = number of fires)")
plt.savefig("figures/scatter_district.png", dpi=300, bbox_inches="tight"); plt.show()

Line plots

In [ ]:
daily = df.groupby("date").size()
plt.figure(figsize=(13, 4))
plt.plot(daily, color="lightgray", label="daily")
plt.plot(daily.rolling(7).mean(), label="7-day moving average")
plt.ylabel("number of fires"); plt.title("Daily fire responses (2024–2025)"); plt.legend()
plt.savefig("figures/line_daily.png", dpi=300, bbox_inches="tight"); plt.show()
print(daily.sort_values(ascending=False).head())

monthly = df.groupby(["year", "month"]).size().unstack(0)
monthly.plot(marker="o", figsize=(9, 4))
plt.xticks(range(1, 13)); plt.xlabel("month"); plt.ylabel("number of fires")
plt.title("Monthly fires: 2024 vs 2025")
plt.savefig("figures/line_monthly.png", dpi=300, bbox_inches="tight"); plt.show()

*Preprocessing & Statistical Analysis*

In [ ]:
# 1. Final dataset check
print("Shape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes)

# duplicate check
n_dup = df.duplicated().sum()
print("\nFully duplicated rows:", n_dup)

# key-based duplicate check: same date, time, district, type and cause
key = ["date", "dispatch_time", "district", "fire_type", "fire_cause"]
print("Duplicates on key columns:", df.duplicated(subset=key).sum())

# sanity check: the translations from Setup worked
print("\nfire_result NaN:", df["fire_result"].isna().sum())
print("region NaN:", df["region"].isna().sum())

***1. Delete duplicates***

In [ ]:
# remove exact duplicates; keep key-duplicates (they differ in other attributes)
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Before:", df.shape, " After:", df_clean.shape)

***2. Attribute Selection***

In [ ]:
human_death = ["male_deaths", "female_deaths", "firefighter_deaths"]
human_inj   = ["male_injured", "female_injured", "firefighter_injured"]
animal      = ["cattle_deaths", "sheep_goat_deaths", "poultry_deaths", "pet_deaths", "other_animal_deaths"]

# how sparse are the casualty columns? (% of fires with a non-zero value)
print("% of fires with value > 0:")
print((df_clean[human_death + human_inj + animal] > 0).mean().mul(100).round(2))

# aggregate sparse columns into totals
df_clean["human_deaths"]  = df_clean[human_death].sum(axis=1)
df_clean["human_injured"] = df_clean[human_inj].sum(axis=1)
df_clean["animal_deaths"] = df_clean[animal].sum(axis=1)

# drop raw/redundant columns
drop_cols = ["date", "arrival_time", "dispatch_time"] + human_death + human_inj + animal
df_sel = df_clean.drop(columns=drop_cols)
print("\nSelected shape:", df_sel.shape)
print(df_sel.columns.tolist())

***3. Examine the missing values***

In [ ]:
miss = df_sel.isna().sum()
miss = miss[miss > 0].to_frame("missing_count")
miss["missing_%"] = (miss["missing_count"] / len(df_sel) * 100).round(2)
print(miss)

# do resource columns contain explicit zeros? (helps decide what NaN means)
print()
for c in ["water_m3", "foam_kg", "dry_powder_kg"]:
    print(f"{c}: zeros = {(df_sel[c] == 0).sum()}, NaN = {df_sel[c].isna().sum()}")

# is building_type missing only for non-structure fires?
print("\nFire types where building_type is MISSING:")
print(df_sel.loc[df_sel["building_type"].isna(), "fire_type"].value_counts().head(8))
print("\nFire types where building_type is FILLED:")
print(df_sel.loc[df_sel["building_type"].notna(), "fire_type"].value_counts().head(8))

***Filling Empty Values***

In [ ]:
before = df_sel.isna().sum()

# building_type: structurally missing (no building in grass/garbage/vehicle fires)
df_sel["building_type"] = df_sel["building_type"].fillna("Not applicable")

# arrival_min: median within the same district (arrival depends on location)
df_sel["arrival_min"] = df_sel["arrival_min"].fillna(
    df_sel.groupby("district")["arrival_min"].transform("median"))

# water / dry powder: median within the same fire type (usage depends on fire type)
for c in ["water_m3", "dry_powder_kg"]:
    df_sel[c] = df_sel[c].fillna(df_sel.groupby("fire_type")[c].transform("median"))
    df_sel[c] = df_sel[c].fillna(df_sel[c].median())   # fallback if a group is all-NaN

after = df_sel.isna().sum()
report = pd.DataFrame({"before": before, "after": after})
report = report[report["before"] > 0]
print(report)
print("\nTotal missing after imputation:", df_sel.isna().sum().sum())

***4. Correlation Matrix***

In [ ]:
df_sel["fire_severity"] = df_sel["fire_result"].cat.codes   # 0 = early, 1 = partial, 2 = complete

corr_cols = ["arrival_min", "water_m3", "foam_kg", "dry_powder_kg",
             "human_deaths", "human_injured", "animal_deaths", "fire_severity"]
corr = df_sel[corr_cols].corr(method="pearson")
print(corr.round(2))

***5. HeatMap***

In [ ]:
!apt-get -qq update
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended cm-super dvipng

In [ ]:
# 4b. District-level correlation (aggregation removes zero-inflation noise)
import matplotlib as mpl

# --- District-level aggregates ---
dist = df_sel.groupby("district").agg(
    n_fires        = ("fire_type", "size"),
    median_arrival = ("arrival_min", "median"),
    damaged_pct    = ("fire_severity", lambda s: (s > 0).mean() * 100),
    mean_water     = ("water_m3", "mean"),
    rural_pct      = ("region", lambda s: (s == "Rural").mean() * 100),
    structure_pct  = ("building_type", lambda s: (s != "Not applicable").mean() * 100))
dist["casualties_per100"] = (df_sel.groupby("district")[["human_deaths", "human_injured"]]
                             .sum().sum(axis=1) / dist["n_fires"] * 100)
dist = dist[dist["n_fires"] >= 30]          # drop districts with too few fires
print("Districts used:", len(dist))

# --- Correlation with paper-consistent labels ---
pretty = {
    "n_fires":           "Number of Fires",
    "median_arrival":    "Median Arrival Time",
    "damaged_pct":       "Damaged Fires (%)",
    "mean_water":        "Mean Water Amount",
    "rural_pct":         "Rural Region (%)",
    "structure_pct":     "Structure Fires (%)",
    "casualties_per100": "Casualties per 100 Fires",
}
corr_d = dist.corr(method="pearson")
corr_d_pretty = corr_d.rename(index=pretty, columns=pretty)

# --- Heatmap (PNG / PDF) ---
plt.figure(figsize=(9, 7.5))
ax = sns.heatmap(corr_d_pretty, annot=True, fmt=".2f", cmap="coolwarm",
                 vmin=-1, vmax=1, square=True, linewidths=0.5,
                 cbar_kws={"shrink": 0.8})
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig("figures/correlation_district.png", dpi=300, bbox_inches="tight")
plt.savefig("figures/correlation_district.pdf", bbox_inches="tight")
plt.show()

# --- LaTeX (colored square-cell table) export ---
def tex_escape(s):
    return s.replace("\\", r"\textbackslash{}").replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

def corr_to_latex(corr, path, cmap="coolwarm", lower=False,
                  cell="2.6em", label_width="8.5em", white_thr=0.6):
    cm = mpl.colormaps[cmap]
    norm = mpl.colors.Normalize(vmin=-1, vmax=1)
    names = [tex_escape(c) for c in corr.columns]
    n = len(names)

    lines = [
        "{%",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{0pt}",
        r"\renewcommand{\arraystretch}{1}",
        r"\ifdefined\cs\else\newlength{\cs}\fi",
        rf"\setlength{{\cs}}{{{cell}}}%",
        r"\providecommand{\cc}[2]{\cellcolor[RGB]{#1}\parbox[c][\cs][c]{\cs}{\centering #2}}",
        r"\providecommand{\cw}[2]{\cc{#1}{\color{white}#2}}",
        rf"\providecommand{{\vl}}[1]{{\makebox[\cs]{{\rotatebox{{90}}{{\makebox[{label_width}][r]{{#1}}}}}}}}",
        r"\begin{tabular}{r@{\hspace{4pt}}" + "c" * n + "}",
    ]
    for i, row in enumerate(names):
        cells = [row]
        for j in range(n):
            if lower and j > i:
                cells.append("")
                continue
            v = corr.iloc[i, j]
            r, g, b, _ = [int(round(x * 255)) for x in cm(norm(v))]
            macro = r"\cw" if abs(v) > white_thr else r"\cc"
            cells.append(rf"{macro}{{{r},{g},{b}}}{{{v:.2f}}}")
        end = r" \\[2pt]" if i == n - 1 else r" \\"
        lines.append(" & ".join(cells) + end)
    lines.append(" & " + " & ".join(rf"\vl{{{c}}}" for c in names) + r" \\")
    lines.append(r"\end{tabular}%")
    lines.append("}")

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved:", path)

corr_to_latex(corr_d_pretty, "figures/correlation_district.tex")

In [ ]:
# 4c. Association between categorical attributes (Cramér's V)
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    return np.sqrt(chi2 / (n * (min(table.shape) - 1)))

cat_cols = ["fire_type", "fire_cause", "fire_result", "building_type", "region", "district"]
cv = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        cv.loc[a, b] = cramers_v(df_sel[a], df_sel[b])

pretty_cat = {"fire_type": "Fire Type", "fire_cause": "Fire Cause", "fire_result": "Fire Result",
              "building_type": "Building Type", "region": "Region", "district": "District"}
cv_pretty = cv.rename(index=pretty_cat, columns=pretty_cat)

# --- Heatmap (PNG / PDF), same style as the district matrix ---
plt.figure(figsize=(7.5, 6.5))
ax = sns.heatmap(cv_pretty, annot=True, fmt=".2f", cmap="Reds",
                 vmin=0, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig("figures/cramers_v.png", dpi=300, bbox_inches="tight")
plt.savefig("figures/cramers_v.pdf", bbox_inches="tight")
plt.show()

# --- LaTeX export: colored cells, 45° labels, color bar (works for any number of matrices) ---
def tex_escape(s):
    return s.replace("\\", r"\textbackslash{}").replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

def matrix_to_latex(mat, path, cmap="coolwarm", vmin=-1, vmax=1, ticks=None,
                    cell="2.6em", white_thr=0.6, prefix="cmA"):
    cm = mpl.colormaps[cmap]
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    rgb = lambda x: ",".join(str(int(round(c * 255))) for c in cm(x)[:3])
    names = [tex_escape(c) for c in mat.columns]
    n = len(names)
    longest = max(names, key=len)
    if ticks is None:
        ticks = np.linspace(vmin, vmax, 5)

    L = ["{%", r"\scriptsize%", r"\setlength{\tabcolsep}{0pt}%", r"\renewcommand{\arraystretch}{1}%",
         r"\ifdefined\cs\else\newlength{\cs}\fi\setlength{\cs}{" + cell + "}%",
         r"\ifdefined\lw\else\newlength{\lw}\fi\settowidth{\lw}{" + longest + "}%",
         r"\providecommand{\cc}[2]{\cellcolor[RGB]{#1}\parbox[c][\cs][c]{\cs}{\centering #2}}%",
         r"\providecommand{\cw}[2]{\cc{#1}{\color{white}#2}}%",
         r"\providecommand{\vl}[1]{\makebox[\cs]{\rule[-0.75\lw]{0pt}{0pt}"
         r"\tikz[baseline=0pt,overlay]\node[rotate=45,anchor=north east,inner sep=1pt] at (0.15\cs,-2pt) {#1};}}%"]
    for k, s in enumerate(np.linspace(0, 1, 9)):                 # color bar stops
        L.append(rf"\definecolor{{{prefix}{k}}}{{RGB}}{{{rgb(s)}}}%")

    top, bot, seg = 0.5, 0.5 - n, n / 8                           # bar spans all n rows
    bar = [r"\raisebox{\fontdimen22\textfont2}[0pt][0pt]{%",
           r"\begin{tikzpicture}[x=\cs,y=\cs,baseline=0pt]",
           rf"\begin{{scope}}\clip (0,{bot}) rectangle (0.35,{top});"]
    for k in range(8):
        y0, y1 = bot + seg * k - 0.01, bot + seg * (k + 1) + 0.01
        bar.append(rf"\shade[bottom color={prefix}{k}, top color={prefix}{k+1}] "
                   rf"(0,{y0:.3f}) rectangle (0.35,{y1:.3f});")
    bar.append(r"\end{scope}")
    for t in ticks:
        y = bot + (t - vmin) / (vmax - vmin) * n
        bar.append(rf"\draw[line width=0.3pt] (0.35,{y:.3f}) -- ++(0.08,0) "
                   rf"node[right,inner sep=1.5pt]{{{t:.2f}}};")
    bar.append(r"\end{tikzpicture}}")

    L.append(r"\begin{tabular}{r@{\hspace{4pt}}" + "c" * n + r"@{\hspace{4pt}}l}")
    for i, row in enumerate(names):
        cells = [row]
        for j in range(n):
            v = mat.iloc[i, j]
            macro = r"\cw" if abs(v) > white_thr else r"\cc"
            cells.append(rf"{macro}{{{rgb(norm(v))}}}{{{v:.2f}}}")
        if i == 0:
            cells.append("\n".join(bar))
        L.append(" & ".join(cells) + (r" \\[2pt]" if i == n - 1 else r" \\"))
    L.append(" & " + " & ".join(rf"\vl{{{c}}}" for c in names) + r" \\")
    L += [r"\end{tabular}%", "}"]

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(L))
    print("Saved:", path)

# both matrices with the same style (different color prefixes so they don't clash)
matrix_to_latex(corr_d_pretty, "figures/correlation_district.tex",
                cmap="coolwarm", vmin=-1, vmax=1, prefix="cwd")
matrix_to_latex(cv_pretty, "figures/cramers_v.tex",
                cmap="Reds", vmin=0, vmax=1, ticks=[0, 0.25, 0.5, 0.75, 1], prefix="rdv")

***6. Discretization***

In [ ]:
# 6. Discretization: arrival time -> Short / Medium / Long (equal-frequency bins)
import matplotlib as mpl

df_sel["arrival_cat"], edges = pd.qcut(df_sel["arrival_min"], q=3,
                                       labels=["Short", "Medium", "Long"], retbins=True)
print("Bin edges (minutes):", edges.round(1))
print(df_sel["arrival_cat"].value_counts().sort_index())

# damage rate per arrival category, separately for urban and rural fires
sub = df_sel[df_sel["region"].isin(["Urban", "Rural"])]
damage = (sub.groupby(["arrival_cat", "region"], observed=True)["fire_severity"]
             .apply(lambda s: (s > 0).mean() * 100).unstack())[["Urban", "Rural"]]
counts = pd.crosstab(sub["arrival_cat"], sub["region"])[["Urban", "Rural"]]
print("\n% partially/completely burned:\n", damage.round(1))
print("\nNumber of fires:\n", counts)

# --- PNG (for the notebook / GitHub) ---
colors = ["tab:blue", "tab:green"]
ax = damage.plot(kind="bar", figsize=(8, 4.5), rot=0, color=colors)
ax.set_xlabel("Arrival Time Category")
ax.set_ylabel(r"Damaged Fires (\%)")          # \% : '%' is a comment character in LaTeX
ax.legend(title="Region")
for c in ax.containers: ax.bar_label(c, fmt="%.1f", fontsize=8)
plt.tight_layout()
plt.savefig("figures/discretization_arrival.png", dpi=300, bbox_inches="tight")
plt.show()

# --- LaTeX (pgfplots) export for the report ---
def bars_to_latex(data, path, xlabel, ylabel, legend_title=None, colors=None, decimals=1):
    cats = [str(c) for c in data.index]
    ymax = float(np.ceil(data.values.max() * 1.15))      # headroom for the value labels
    L = ["{%", r"\begin{tikzpicture}", r"\begin{axis}[",
         r"  width=\columnwidth, height=0.62\columnwidth,",
         rf"  ybar, bar width=9pt, ymin=0, ymax={ymax:g},",
         r"  symbolic x coords={" + ",".join(cats) + "}, xtick=data,",
         r"  enlarge x limits=0.25,",
         r"  xlabel={" + xlabel + "}, ylabel={" + ylabel + "},",
         r"  tick label style={font=\scriptsize}, label style={font=\small},",
         r"  nodes near coords={\pgfmathprintnumber[fixed,fixed zerofill,precision="
             + str(decimals) + r"]{\pgfplotspointmeta}},",
         r"  every node near coord/.append style={font=\tiny},",
         r"  legend style={at={(0.02,0.98)}, anchor=north west, font=\scriptsize},",
         r"  legend cell align=left,",
         "]"]
    if legend_title:
        L.append(r"\addlegendimage{empty legend}\addlegendentry{\textbf{" + legend_title + "}}")
    for col, c in zip(data.columns, colors):
        r, g, b = [int(round(x * 255)) for x in mpl.colors.to_rgb(c)]
        coords = " ".join(f"({cat},{v:.{decimals}f})" for cat, v in zip(cats, data[col]))
        L.append(rf"\addplot[fill={{rgb,255:red,{r};green,{g};blue,{b}}}, draw=none] "
                 rf"coordinates {{{coords}}};")
        L.append(r"\addlegendentry{" + str(col) + "}")
    L += [r"\end{axis}", r"\end{tikzpicture}%", "}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(L))
    print("Saved:", path)

bars_to_latex(damage, "figures/discretization_arrival.tex",
              xlabel="Arrival Time Category", ylabel=r"Damaged Fires (\%)",
              legend_title="Region", colors=colors)

In [ ]:
# 6c. Share of fires reached late (> 6 min) per district
from matplotlib.patches import Patch

d = df_sel.groupby("district").agg(
    n_fires    = ("fire_type", "size"),
    long_pct   = ("arrival_cat", lambda s: (s == "Long").mean() * 100),
    damage_pct = ("fire_severity", lambda s: (s > 0).mean() * 100),
    rural_pct  = ("region", lambda s: (s == "Rural").mean() * 100))
d = d[d["n_fires"] >= 30].sort_values("long_pct")          # smallest at the bottom
d["type"] = np.where(d["rural_pct"] > 50, "Mostly rural", "Mostly urban")
print(d.sort_values("long_pct", ascending=False).round(1).to_string())

# --- Turkish-aware title case + LaTeX-safe Turkish letters ---
def tr_title(s):                                   # "ÇİĞLİ" -> "Çiğli"
    s = s.replace("I", "ı").replace("İ", "i").lower()
    return {"i": "İ", "ı": "I"}.get(s[0], s[0].upper()) + s[1:]

TEX_TR = {"ç": r"\c{c}", "Ç": r"\c{C}", "ğ": r"\u{g}", "Ğ": r"\u{G}", "ı": r"{\i}", "İ": r"\.{I}",
          "ö": r'\"{o}', "Ö": r'\"{O}', "ş": r"\c{s}", "Ş": r"\c{S}", "ü": r'\"{u}', "Ü": r'\"{U}'}
def tex_tr(s):
    return "".join(TEX_TR.get(ch, ch) for ch in s)

names_plain = [tr_title(x) for x in d.index]
names_tex   = [tex_tr(x) for x in names_plain]
palette = {"Mostly urban": "tab:blue", "Mostly rural": "tab:green"}

# --- PNG (for the notebook / GitHub) ---
y = np.arange(len(d))
fig, ax = plt.subplots(figsize=(8, 9))
bars = ax.barh(y, d["long_pct"], color=d["type"].map(palette))
ax.set_yticks(y)
ax.set_yticklabels(names_tex if plt.rcParams["text.usetex"] else names_plain)
ax.bar_label(bars, fmt="%.0f", fontsize=7, padding=2)
ax.set_xlabel(r"Fires Reached in More Than 6 Minutes (\%)")
ax.set_xlim(0, 105)
ax.legend(handles=[Patch(color=c, label=l) for l, c in palette.items()], loc="lower right")
plt.tight_layout()
plt.savefig("figures/late_arrival_by_district.png", dpi=300, bbox_inches="tight")
plt.show()

# --- LaTeX (pgfplots) export for the report ---
def district_bars_to_latex(d, labels, path, palette):
    n = len(d)
    L = ["{%", r"\begin{tikzpicture}", r"\begin{axis}[",
         rf"  xbar, bar width=6pt, width=\columnwidth, height={0.36 * n:.1f}cm,",
         rf"  xmin=0, xmax=105, ytick={{0,...,{n - 1}}},",
         r"  yticklabels={" + ",".join("{" + l + "}" for l in labels) + "},",
         r"  enlarge y limits={abs=0.6}, ytick style={draw=none},",
         r"  xlabel={Fires Reached in More Than 6 Minutes (\%)},",
         r"  tick label style={font=\scriptsize}, label style={font=\small},",
         r"  point meta=x,",
         r"  nodes near coords={\pgfmathprintnumber[fixed,precision=0]{\pgfplotspointmeta}},",
         r"  nodes near coords align=horizontal,",
         r"  every node near coord/.append style={font=\tiny},",
         r"  legend style={at={(0.98,0.02)}, anchor=south east, font=\scriptsize},",
         r"  legend cell align=left,",
         "]"]
    for label, color in palette.items():
        idx = [i for i, t in enumerate(d["type"]) if t == label]
        if not idx:
            continue
        r, g, b = [int(round(x * 255)) for x in mpl.colors.to_rgb(color)]
        coords = " ".join(f"({d['long_pct'].iloc[i]:.1f},{i})" for i in idx)
        L.append(rf"\addplot[fill={{rgb,255:red,{r};green,{g};blue,{b}}}, draw=none, bar shift=0pt] "
                 rf"coordinates {{{coords}}};")
        L.append(r"\addlegendentry{" + label + "}")
    L += [r"\end{axis}", r"\end{tikzpicture}%", "}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(L))
    print("Saved:", path)

district_bars_to_latex(d, names_tex, "figures/late_arrival_by_district.tex", palette)

***7. Normalization (Min-Max) and Standardization (z-score)***

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scale_cols = ["arrival_min", "water_m3", "foam_kg", "dry_powder_kg",
              "human_deaths", "human_injured", "animal_deaths"]

df_norm = df_sel.copy()
df_norm[scale_cols] = MinMaxScaler().fit_transform(df_sel[scale_cols])

df_std = df_sel.copy()
df_std[scale_cols] = StandardScaler().fit_transform(df_sel[scale_cols])

summary = pd.concat({
    "Original":        df_sel[scale_cols].agg(["min", "max", "mean", "std"]).T,
    "Min-Max":         df_norm[scale_cols].agg(["min", "max", "mean", "std"]).T,
    "Standardization": df_std[scale_cols].agg(["min", "max", "mean", "std"]).T}, axis=1)
print(summary.round(3))

# --- Histograms (PNG): same variable, three scales; the shape does not change ---
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (name, d) in zip(axes, [("Original (min)", df_sel), ("Min-Max [0, 1]", df_norm),
                                ("Standardized (z-score)", df_std)]):
    ax.hist(d["arrival_min"], bins=60, edgecolor="white")
    ax.set_title(f"Arrival Time: {name}"); ax.set_yscale("log")
    ax.set_ylabel("Number of Fires (log)")
plt.tight_layout()
plt.savefig("figures/scaling_arrival.png", dpi=300, bbox_inches="tight")
plt.show()

# --- LaTeX (pgfplots) version of the three histograms ---
def hists_to_latex(series_list, titles, path, bins=60, color="tab:blue"):
    r, g, b = [int(round(x * 255)) for x in mpl.colors.to_rgb(color)]
    L = ["{%", r"\begin{tikzpicture}"]
    for k, (s, title) in enumerate(zip(series_list, titles)):
        counts, edges = np.histogram(s.dropna(), bins=bins)
        centers = (edges[:-1] + edges[1:]) / 2
        width = edges[1] - edges[0]
        coords = " ".join(f"({x:.6g},{c})" for x, c in zip(centers, counts) if c > 0)
        pos = "" if k == 0 else rf"at={{(h{k-1}.south east)}}, anchor=south west, xshift=0.9cm,"
        ylab = r"ylabel={Number of Fires}," if k == 0 else ""
        L += [r"\begin{axis}[",
              rf"  name=h{k}, {pos}",
              r"  scale only axis, width=0.25\textwidth, height=3.2cm,",
              r"  ymode=log, log origin=infty, ymin=0.5,",
              rf"  title={{{title}}}, xlabel={{Arrival Time}}, {ylab}",
              r"  title style={font=\small}, label style={font=\small},",
              r"  tick label style={font=\scriptsize},",
              "]",
              rf"\addplot[ybar, bar width={width:.6g}, fill={{rgb,255:red,{r};green,{g};blue,{b}}}, "
              rf"draw=white, line width=0.2pt] coordinates {{{coords}}};",
              r"\end{axis}"]
    L += [r"\end{tikzpicture}%", "}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(L))
    print("Saved:", path)

hists_to_latex([df_sel["arrival_min"], df_norm["arrival_min"], df_std["arrival_min"]],
               ["Original (min)", "Min-Max [0, 1]", r"Standardized ($z$-score)"],
               "figures/scaling_arrival.tex")
# save the standardized table so Person 3 can use it directly for PCA
df_std.to_csv("standardized_data.csv", index=False)

***8. One-hot encoding***

In [ ]:
counts = df_sel["fire_type"].value_counts()
print("Number of fire types:", len(counts))

# group rare types (< 1% of fires) into "Other" to avoid many near-empty columns
rare = counts[counts / len(df_sel) < 0.01].index
df_enc = df_sel.copy()
df_enc["fire_type_grouped"] = df_enc["fire_type"].where(~df_enc["fire_type"].isin(rare), "Other")
print("Types merged into 'Other':", len(rare))
print("Categories after grouping:", df_enc["fire_type_grouped"].nunique())

onehot = pd.get_dummies(df_enc["fire_type_grouped"], prefix="type", dtype=int)
df_enc = pd.concat([df_enc, onehot], axis=1)

print("\nShape before:", df_sel.shape, " after:", df_enc.shape)
print("\nRecords per new column:")
print(onehot.sum().sort_values(ascending=False))
print("\nEvery row has exactly one 1:", (onehot.sum(axis=1) == 1).all())

# example rows: original value next to its encoded columns
print(df_enc[["fire_type"] + onehot.columns.tolist()].sample(6, random_state=1).to_string())

In [ ]:
df_sel.to_csv("preprocessed_data.csv", index=False)

In [ ]:
print("Columns (original):", d24.shape[1] - 1)   # minus the YIL column we added
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Unique fire causes:", df["fire_cause"].nunique())
print("Unique districts:", df["district"].nunique())